# SIMD 与 SIMT 混合编程：核函数、VF 函数与用户界面

## 概述

上一节了解了混合编程"以 SIMD 为主、SIMT 为辅"的设计原则。本节从代码层面学习混合编程是如何组织的：核函数如何定义与调用，Vector Function（VF）这一软件概念如何让 SIMD、SIMT 硬件单元协同工作，混合编程涉及的各类函数遵循怎样的调用层级约束，并从用户界面角度对比混合编程场景与纯 SIMT 编程场景在代码写法上的主要差异。

### 前置要求

- 已学习 3.5.1 混合编程概述，理解 SIMD 与 SIMT 的算力分工。
- 已学习 3.3.2 SIMD 核函数、3.4.3 SIMT 核函数，理解各自场景下核函数的定义和调用方式。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 写出混合编程场景下核函数的定义语法，说明核函数调用符中各配置参数的含义。
- 理解 Vector Function（VF）的概念，说明 SIMD VF 和 SIMT VF 的区别与 `asc_vf_call` 的调用方式。
- 列出混合编程涉及的各类函数修饰符，说明它们的调用层级关系。
- 对比纯 SIMT 编程场景与混合编程场景下核函数写法、参数修饰和编译选项上的差异。

### 小节内容

- 核函数的定义与调用
- 混合编程中的函数类型与调用层级
- 纯 SIMT 编程场景与混合编程场景的用户界面对比

## 核函数的定义与调用

核函数是 SIMD 与 SIMT 混合编程的 Device 侧入口函数，负责协调整个算子的执行流程，包括 VF 的调度和调用。下面以 AI Vector 核内的混合编程为例，函数定义语法如下：

```text
__global__ __vector__ void kernel_name(__gm__ type* param1, __gm__ type* param2, ...);
```
其中，`__global__` 标识核函数，表明可在 Host 侧通过 `<<<...>>>` 调用。`__vector__` 标识函数是在 Device 侧 AI Vector 上执行。

核函数的调用是通过 `<<<...>>>` 核函数调用符在 Host 侧调用，语法如下：

```text
kernel_name<<<num_blocks, dyn_ub_size, stream>>>(args...);
```

核函数调用符内的配置参数说明如下：

| 参数 | 类型 | 说明 | 约束 |
| --- | --- | --- | --- |
| `num_blocks` | `uint32_t` | 设置核函数启用的核数 | 取值范围[1, 65535] |
| `dyn_ub_size` | `uint32_t` | 指定动态内存大小，单位为字节 | 不超过最大可配置值：256KB - 8KB - 32KB - 静态内存 |
| `stream` | `aclrtStream` | 用于维护异步操作执行顺序 | 无 |

## 混合编程中的函数类型与调用层级

SIMD 与 SIMT 混合编程涉及多种类型的函数，它们之间遵循严格的调用关系和层级约束。混合编程中涉及的函数类型有：

| 修饰符 | 函数功能 | 调用方式 |
| --- | --- | --- |
| `__global__ __aicore__` | 算子入口，协调 VF 执行。若只有在 AIV 核内执行的 SIMD 与 SIMT 混合编程场景，可使用 `__global__ __vector__` 来标识只启动 AIV 核。 | Host 侧通过 `<<<...>>>` 调用 |
| `__aicore__` | Device 侧辅助函数 | 核函数或同级函数调用 |
| `__simt_vf__` | 线程级并行计算任务 | 通过 SIMT 提供的 `asc_vf_call` 接口调用 |
| `__simd_vf__` | 向量级并行计算任务 | 通过 SIMD 提供的 `asc_vf_call` 接口调用 |
| `__simt_callee__` | SIMT VF 的子函数 | SIMT VF 内部调用 |
| `__simd_callee__` | SIMD VF 的子函数 | SIMD VF 内部调用 |
| `__callee__` | SIMD VF 和 SIMT VF 的公共子函数 | VF 内部调用 |

各层函数间的调用关系为：

![](images/03_05_hybrid/hybrid_function_call_hierarchy.png)

核函数和 `__aicore__` 函数可以通过 `asc_vf_call` 调用 VF，执行相应的计算任务。其中，SIMT VF 函数用于实现线程级并行计算任务，处理不规则访问和复杂控制逻辑，函数内可使用 SIMT 内置变量：threadIdx、blockIdx、blockDim、gridDim。

```text
__simt_vf__ __launch_bounds__(MAX_THREAD_COUNT) inline void function_name(
    __gm__ type* gm_param,
    __ubuf__ type* ubuf_param,
    type scalar_param, ...);
```

上述代码段为 SIMT VF 函数的定义示例，其中的关键修饰符说明如下：

| 修饰符 | 作用 |
| --- | --- |
| `__simt_vf__` | 函数标识符，标识 SIMT VF 函数 |
| `__launch_bounds__(N)` | 指定最大线程数（可选，默认 1024） |
| `inline` | 建议内联，实际是否内联由编译器决定 |
| `__gm__` | 内存空间修饰符，标识内存空间为 GM |
| `__ubuf__` | 内存空间修饰符，标识内存空间为 UB |

使用 `asc_vf_call` 调用 VF 函数时，SIMT VF 函数相比 SIMD VF 函数多一个线程规模配置参数，该参数位于 `asc_vf_call` 的第一个参数位置：

```text
asc_vf_call<simd_func>(arg1, arg2, ...);
uint32_t thread_num = 1024;
asc_vf_call<simt_func>(dim3(thread_num), arg1, arg2, ...);
```

## 纯 SIMT 编程场景与混合编程场景的用户界面对比

前面两部分介绍了混合编程场景下核函数与 VF 函数的定义、调用层级。下面从用户界面角度，对比混合编程场景与纯 SIMT 编程场景在代码写法上的主要差异。


### 纯 SIMT 编程场景的用户界面

在纯 SIMT 编程场景中（例如 Gather、Scatter 等离散访存算子），用户界面如下：

```cpp
// Device侧核函数
__global__ void kernel_name(float* param1, float* param2, ...)
{
    // ...
}

// Host侧：通过<<<>>>调用device侧核函数
int32_t main(int argc, char const *argv[])
{
    // ...
    kernel_name<<<blocks_per_grid, threads_per_block, dyn_ub_size, stream>>>(param1, param2, ...);
    // ...
}
```

其中，核函数使用 `__global__` 标识，并通过 Host 侧 `<<<>>>` 调用。`param1`、`param2` 等核函数指针参数对应 GM 上的数据地址。`blocks_per_grid` 和 `threads_per_block` 为 `dim3` 结构体，分别用于指定网格（grid）和每个线程块（block）的维度与规模，`dyn_ub_size` 表示动态共享内存的大小。

### SIMD 与 SIMT 混合编程场景的用户界面

在 SIMD 与 SIMT 混合编程场景中，编写一个简单的算子与纯 SIMT 编程场景有所不同，用户界面如下：

```cpp
// Device侧SIMT VF函数
__simt_vf__ inline void simt_func(__gm__ float* param3, __ubuf__ float* param4, ...)
{
    // ...
}

__simd_vf__ inline void simd_func(__ubuf__ float* param5, __ubuf__ float* param6, ...)
{
    // ...
}

// Device侧核函数
__global__ __vector__ void kernel_name(__gm__ float* param1, __gm__ float* param2, ...)
{
    
    // ...

    // 执行SIMT VF 函数
    asc_vf_call<simt_func>(dim3(dimx, dimy, dimz), param3, param4, ...);
    // 执行SIMD VF 函数
    asc_vf_call<simd_func>(param5, param6,...);

    // ...
}

// Host侧：通过<<<>>>调用device侧核函数
int32_t main(int32_t argc, char const *argv[])
{
    // ...
    kernel_name<<<num_blocks, dyn_ub_size, stream>>>(param1, param2, ...);
    // ...
}
```

其中，核函数使用 `__global__ __vector__` 标识，表示在 AI Vector 核上执行。核函数内部通过 `asc_vf_call` 调用 VF 函数：SIMT VF 函数和 SIMD VF 函数分别使用 `__simt_vf__`、`__simd_vf__` 修饰，用于区分代码段运行的硬件单元。SIMT VF 函数的参数支持 GM 地址和 UB 地址，SIMD VF 函数的参数仅支持 UB 地址。GM 地址使用 `__gm__` 修饰，UB 地址使用 `__ubuf__` 修饰。多个 VF 函数按照调用顺序串行执行，执行顺序由硬件保证。对于 SIMT VF 函数，线程规模通过 `asc_vf_call` 的第一个参数 `dim3(dimx, dimy, dimz)` 配置。Host 侧启动核函数时，`num_blocks` 表示启用的 AI Vector 核数。

### 两种场景的对比总结

对比 SIMT 编程场景与 SIMD/SIMT 混合编程场景调用 SIMT 硬件单元的用户界面，主要差异如下：

| 对比项 | SIMT 编程场景 | SIMD / SIMT 混合编程场景 |
| --- | --- | --- |
| **SIMT 单元调用方式** | Host 侧通过 `<<<>>>` 直接启动 `__global__` SIMT 核函数 | Host 侧通过 `<<<>>>` 启动 `__global__ __vector__` 核函数，再由 Device 侧通过 `asc_vf_call` 调用 SIMT VF 函数 |
| **线程数配置** | 在 Host 侧 `<<<>>>` 中通过 `threads_per_block` 配置 | 在 Device 侧通过 `asc_vf_call` 的第一个参数 `dim3(dimx, dimy, dimz)` 配置 |
| **Host 侧调用** | `<<<>>>` 共 **4 个参数**，其中 `blocks_per_grid` 为 dim3 结构体表示 grid 维度与规模 | `<<<>>>` 共 **3 个参数**，其中 `num_blocks` 表示核数，不需要配置线程数 |
| **参数修饰** | 普通指针类型 | 根据数据所在存储空间使用 `__gm__`、`__ubuf__` 等内存空间修饰符 |
| **编译选项** | 需要加上 `--enable-simt` 编译选项 | 不需要加上 `--enable-simt` 编译选项 |

至此，SIMD 与 SIMT 混合编程的基础概念、核函数/VF 函数、内存层级和用户界面已经介绍完毕。后续实践章节将以具体算子为例，展示混合编程的开发与性能优化实践。

## 术语速查

<table>
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Vector Function（VF）</td>
      <td>表示在 SIMT 或 SIMD 硬件计算单元上执行的特定功能代码段的软件概念</td>
    </tr>
    <tr>
      <td>asc_vf_call</td>
      <td>核函数或 device 侧函数调用 VF 的接口；调用 SIMT VF 时需在第一个参数传入线程规模 dim3</td>
    </tr>
    <tr>
      <td>PIPE_V</td>
      <td>Vector Function 所属的流水线，与 MTE 搬运流水（PIPE_MTE2/PIPE_MTE3）相互独立、可并行</td>
    </tr>
    <tr>
      <td>__simt_vf__ / __simd_vf__</td>
      <td>分别标识 SIMT VF 和 SIMD VF 函数的修饰符</td>
    </tr>
    <tr>
      <td>--enable-simt</td>
      <td>纯 SIMT 核函数编译时需要的编译选项；混合编程的核函数不需要该选项</td>
    </tr>
    <tr>
      <td>__gm__ / __ubuf__</td>
      <td>混合编程中用于区分参数所在存储空间的内存空间修饰符，分别对应 GM 和 UB</td>
    </tr>
    <tr>
      <td>num_blocks</td>
      <td>混合编程核函数调用符中的参数，表示启用的 AI Vector 核数，不包含线程数配置</td>
    </tr>
  </tbody>
</table>

## 小节小结

本小节学习了混合编程中函数与调用层级相关的知识，并从用户界面角度对比了纯 SIMT 编程场景和 SIMD/SIMT 混合编程场景：

- **核函数**：使用 `__global__ __vector__` 定义，通过 `<<<num_blocks, dyn_ub_size, stream>>>` 从 Host 侧调用，负责协调 VF 的调度。
- **VF 概念**：Vector Function 是表示 SIMT/SIMD 硬件计算单元上执行代码段的软件抽象，通过 `asc_vf_call` 从核函数或 `__aicore__` 函数中调用；VF 属于 `PIPE_V` 流水，可与 MTE 搬运流水并行。
- **函数类型与调用层级**：核函数/`__aicore__` 函数可调用 SIMT VF 和 SIMD VF；VF 内部可调用各自的 `__simt_callee__`/`__simd_callee__` 子函数，也可调用两者共用的 `__callee__` 子函数。
- **核函数标识与调用层级差异**：纯 SIMT 场景 Host 侧直接启动 `__global__` 核函数；混合编程场景 Host 侧启动 `__global__ __vector__` 核函数，再由核函数内部通过 `asc_vf_call` 调用 SIMT VF/SIMD VF。
- **线程数配置位置差异**：纯 SIMT 在 Host 侧 `<<<>>>` 中配置；混合编程在 Device 侧 `asc_vf_call` 的 `dim3` 参数中配置。
- **参数修饰与编译选项差异**：混合编程使用 `__gm__`/`__ubuf__` 区分存储空间，且不需要 `--enable-simt` 编译选项。

下一小节将学习混合编程场景下的内存层级，特别是 UB 空间在静态内存、动态内存和 SIMT Data Cache 之间的划分方式。

## 课后练习

本节介绍了混合编程中核函数、VF 函数的定义方式和调用层级，并从用户界面角度对比了纯 SIMT 编程场景与混合编程场景，请根据学习内容完成以下题目进行自测。

1. （判断题）在混合编程场景下，SIMT VF 函数可以直接被 Host 侧通过 `<<<...>>>` 调用。

2. （单选题）使用 `asc_vf_call` 调用 SIMT VF 函数时，第一个参数通常用于配置什么？  
    A. GM 地址  
    B. UB 地址  
    C. 线程规模 `dim3`  
    D. 流水线类型  

3. （单选题）Vector Function 属于哪种流水线，因而可以与 MTE 搬运并行执行？  
    A. `PIPE_S`  
    B. `PIPE_V`  
    C. `PIPE_MTE2`  
    D. `PIPE_MTE3`  

4. （多选题）以下哪些函数修饰符属于混合编程中定义的函数类型？  
    A. `__simt_vf__`  
    B. `__simd_vf__`  
    C. `__callee__`  
    D. `__host__`  

5. （判断题）在 SIMD 与 SIMT 混合编程场景中，核函数调用符 `<<<>>>` 内需要配置线程数参数 `threads_per_block`。

6. （单选题）在混合编程场景下，SIMT VF 函数的线程规模通过什么方式配置？  
    A. Host 侧 `<<<>>>` 的第二个参数  
    B. `asc_vf_call` 的第一个参数 `dim3(dimx, dimy, dimz)`  
    C. 核函数体内的全局变量  
    D. CMake 编译选项  

7. （单选题）纯 SIMT 核函数编译时通常需要加上哪个编译选项，而混合编程核函数不需要？  
    A. `--npu-arch`  
    B. `--enable-simt`  
    C. `-O2`  
    D. `--enable-vector`  

8. （多选题）以下关于混合编程场景用户界面的说法，哪些是正确的？  
    A. 核函数使用 `__global__ __vector__` 标识  
    B. SIMD VF 函数的参数仅支持 UB 地址  
    C. SIMT VF 函数的参数不能使用 `__gm__` 修饰  
    D. 多个 VF 函数按调用顺序串行执行  

**执行以下代码获取答案。**

In [ ]:
!cat answer/03.05.02_answer.txt
